# Introduction


In this assignment, you will practice building and training Convolutional Neural Networks with Pytorch to solve computer vision tasks.  This assignment includes two sections, each involving different tasks:

(1) Image Classification. Predict image-level category labels on two historically notable image datasets: **CIFAR-10** and **MNIST**.

(2) Image Segmentation. Predict pixel-wise classification (semantic segmentation) on synthetic input images formed by superimposing MNIST images on top of CIFAR images.

You will design your own models in each section and build the entire training/testing pipeline with PyTorch. 
PyTorch provides optimized implementations of the building blocks and additional utilities, both of which will be necessary for experiments on real datasets. It is highly recommended to read the official [documentation](https://pytorch.org/docs/stable/index.html) and [examples](https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html) before starting your implementation. There are some APIs that you'll find useful:
[Layers](http://pytorch.org/docs/stable/nn.html),
[Activations](https://pytorch.org/docs/stable/nn.html#non-linear-activations-weighted-sum-nonlinearity),
[Loss functions](http://pytorch.org/docs/stable/nn.html#loss-functions),
[Optimizers](http://pytorch.org/docs/stable/optim.html)

It is highly recommended to use Google Colab and run the notebook on a GPU node. Check https://colab.research.google.com/ and look for tutorials online. To use a GPU go to Runtime -> Change runtime type and select GPU. 


# (2) Image Segmentation
The task consists of performing pixel-wise classification on a synthetic dataset
of 32x32 RGB images. Each image was generated by placing a MNIST sample (a grayscale
image of a digit between 0 and 9) on top of a CIFAR-10 sample (a RGB image drawn from
one of 10 possible classes). Each image has an accompanying target tensor of size 32x32,
where in each pixel location (i,j) it contains the ground-truth label of the MNIST digit
(ranging from 0 to 9) or of the CIFAR-10 image (ranging from 10 to 19), depending on whether
the (i,j) pixel in the original image belongs to the superposed MNIST image or not. The
metric of interest here is pixel-wise accuracy, which is the fraction of pixels in each image
for which your model predicted the correct class (out of a total of 20 classes, as described 
above).

Note that there are many ways to frame the above task. For example, your CNN can directly
output a 20x32x32 tensor for each input image, representing a distribution over the possible
20 classes for each of the 32x32 pixels. However, note that the problem has a lot of additional
structure: for example, each 32x32 target tensor only has two distinct numbers in it, the label
of the MNIST digit and the label of the CIFAR-10 background image -- accounting for such
structure will make training faster and likely improve your model's final performance. Your
model should be able to achieve around 70% accuracy on the test set when trained for 100 epochs.

To finish this section step by step, you need to:

* Prepare data by building a dataset and data loader. (already provided below)

* Implement training code (6 points) & testing code (6 points), including saving and loading of models.

* Construct a model (12 points) and choose an optimizer (3 points).

* Describe what you did, any additional features you implemented, and/or any graphs you made in training and evaluating your network. Report final test accuracy @100 epochs in a writeup: hw3.pdf (3 points)

In [1]:
import numpy as np
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import sampler
import torchvision
import torchvision.transforms as T
from utils import SegDataset


## Data Preparation:

Setup a Dataset for training and testing.

Datasets load single training examples one a time, so we practically wrap each Dataset in a DataLoader, which loads a data batch in parallel.

In [2]:
seg_train = SegDataset('./data', train=True, transform=None)
loader_train = DataLoader(seg_train, batch_size=64, shuffle=True)
seg_test = SegDataset('./data', train=False, transform=None)
loader_test = DataLoader(seg_test, batch_size=64, shuffle=False)

## Design/choose your own model structure (12 points) and optimizer (3 points).
You might want to adjust following configurations for better performance:

(1) Network architecture:
- You can borrow some ideas from existing convnets design, e.g., [ResNet](https://arxiv.org/abs/1512.03385) where
the input from the previous layer is added to the output, or [UNet](https://arxiv.org/pdf/1505.04597.pdf) where you can stack intermediate features from previous layers. 
- Note: Do not **directly copy** an existing network design.

(2) Architecture hyperparameters:
- Filter size, number of filters, and number of layers (depth). Make careful choices to tradeoff computational efficiency and accuracy.
- Pooling vs. Strided Convolution
- Batch normalization
- Choice of non-linear activation

(3) Choice of optimizer (e.g., SGD, Adam, Adagrad, RMSprop) and associated hyperparameters (e.g., learning rate, momentum).


In [3]:
class myNet(nn.Module):
    def __init__(self):
        super(myNet, self).__init__()

        def block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                
                nn.Conv2d(out_ch, out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True)
            )

        #contraction
        self.process1 = block(3, 64)
        self.process2 = block(64, 128)
        self.process3 = block(128, 256)
        self.process4 = block(256, 512)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = block(512, 1024)

        #expansion

        self.expand4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.process5 = block(1024, 512)

        self.expand3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.process6 = block(512, 256)

        self.expand2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.process7 = block(256, 128)

        self.expand1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.process8 = block(128, 64)

        self.out = nn.Conv2d(64, 20, 1)

    def save(self):
        torch.save(self.state_dict(), "segmentation_model.pth")

    def load(self, device = "cpu"):
        self.load_state_dict(torch.load("segmentation_model.pth", map_location=device))
        return self


    def forward(self, x):
        b1 = self.process1(x)
        b2 = self.process2(self.pool(b1))
        b3 = self.process3(self.pool(b2))
        b4 = self.process4(self.pool(b3))

        transition = self.bottleneck(self.pool(b4))

        b5 = self.process5(torch.cat([self.expand4(transition), b4], dim=1))
        b6 = self.process6(torch.cat([self.expand3(b5), b3], dim=1))
        b7 = self.process7(torch.cat([self.expand2(b6), b2], dim=1))
        b8 = self.process8(torch.cat([self.expand1(b7), b1], dim=1))

        return self.out(b8)

## Training (6 points)

Train a model on the given dataset using the PyTorch Module API.

Inputs:
- loader_train: The loader from which train samples will be drawn from.
- loader_test: The loader from which test samples will be drawn from
- model: A PyTorch Module giving the model to train.
- optimizer: An Optimizer object we will use to train the model
- epochs: (Optional) A Python integer giving the number of epochs to train for

Returns: Nothing, but prints model accuracies during training.

In [ ]:
def train(loader_train, loader_test, model, optimizer, epochs=100):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    criterion = nn.CrossEntropyLoss() # choose your loss here, if needed
    
    for e in range(epochs):
        model.train()
        for t, (x, y) in enumerate(loader_train):
            x = x.to(device)
            y = y.to(device)
            results = model(x)
            loss = criterion(results, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if t % 100 == 0:
                print('Epoch %d, Iteration %d, loss = %.4f' % (e, t, loss.item()))
        test(loader_test, model)

## Testing (6 points)
Test a model using the PyTorch Module API.

Inputs:
- loader: The loader from which test samples will be drawn from.
- model: A PyTorch Module giving the model to test.

Returns: Nothing, but prints model accuracies during training.

In [ ]:
def test(loader, model):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_correct = 0
    num_samples = 0
    model.eval() # set model to evaluation mode
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            results = model(x)
            preds = results.argmax(dim=1)  # Get class prediction for each pixel
            
            # Compare predictions with targets pixel-wise
            num_correct += (preds == y).sum().item() #condensed 
            num_samples += y.numel()
            
        acc = float(num_correct) / num_samples
        
        print('Eval %d / %d correct (%.2f)' % (num_correct, num_samples, 100 * acc))

Describe your design details in the writeup hw3.pdf. (3 points)

Finish your model and optimizer below.

In [ ]:
lr = 1e-4
momentum = 0.9
weight_decay = 1e-4

model = myNet()
optimizer = optim.Adam(model.parameters(), lr = lr, weight_decay = weight_decay)#changed
train(loader_train, loader_test, model, optimizer, epochs=100)